# Step 1 Baseline Comparison

This notebook evaluates the stopped self-forcing baseline without touching the test split. It runs two deliberately separated comparisons:

1. **Direct multipart comparison:** self-forcing `best`, self-forcing `best_rollout`, and the earlier 6K q0-only planner predict the same 16 causal multipart IDs. Their CE and accuracy are directly comparable.
2. **Released SentiAvatar comparison:** `checkpoints/llm` predicts four legacy whole-body RVQ IDs from action text and HuBERT tokens. Its scores are compared with uniform and persistence inside its own representation. Raw token accuracy must not be treated as directly equivalent to multipart accuracy.

The shared cross-representation gate is decoded motion followed by the appropriate frozen Step 2; that is intentionally not relabelled as token evaluation here.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'motion_generation').is_dir() and (candidate / 'checkpoints').is_dir():
            return candidate
    raise RuntimeError('Could not locate the sentiAvatar-sandbox project root')

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / 'motion_generation/outputs/step1_baseline_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('project:', PROJECT_ROOT)
print('python:', sys.executable)

## 1. Protocol controls

The pilot uses all 635 clips for teacher forcing and the same deterministic 128 validation clips for both generated-rollout systems. After inspecting the pilot, set both rollout limits to `0` for complete-validation reporting.

In [ ]:
RUN_MULTIPART = True
RUN_RELEASED_SENTIAVATAR = True
SUBSET_SEED = 42
MULTIPART_TEACHER_MAX_CLIPS = 0   # 0 = all validation clips
MULTIPART_ROLLOUT_MAX_CLIPS = 128 # change to 0 for final full validation
LEGACY_ROLLOUT_MAX_CLIPS = 128    # must match multipart pilot
MULTIPART_ROLLOUT_BATCH = 8
LEGACY_BATCH = 8
DATA_DIR = PROJECT_ROOT / 'SuSuInterActs/SuSuInterActs'

CHECKPOINTS = {
    'self_forcing_best': PROJECT_ROOT / 'checkpoints/step1_multipart_fixed_gap3_self_forcing_q0q3_full/best',
    'self_forcing_best_rollout': PROJECT_ROOT / 'checkpoints/step1_multipart_fixed_gap3_self_forcing_q0q3_full/best_rollout',
    'q0_6k_final': PROJECT_ROOT / 'checkpoints/step1_multipart_fixed_gap3_main6000/final',
    'released_sentiavatar': PROJECT_ROOT / 'checkpoints/llm',
}
LEGACY_PATHS = {
    'legacy_motion_tokens': DATA_DIR / 'motion_token_data',
    'legacy_hubert_tokens': DATA_DIR / 'audio_tokens_hubert_layer9_fps10',
    'validation_split': DATA_DIR / 'split/val_file_list.txt',
    'motion_text': DATA_DIR / 'text_data/motion2text.json',
}
preflight = pd.DataFrame([
    {'item': label, 'path': str(path), 'exists': path.exists()}
    for label, path in {**CHECKPOINTS, **LEGACY_PATHS}.items()
])
display(preflight)
required = dict(CHECKPOINTS)
if RUN_RELEASED_SENTIAVATAR:
    required.update(LEGACY_PATHS)
missing = {label: path for label, path in required.items() if not path.exists()}
if missing:
    raise FileNotFoundError(
        'Missing evaluation prerequisites: ' + repr(missing) + '\n'
        'If legacy exports are missing, run preprocess_data.py --audio and --motion '
        'with --data_dir SuSuInterActs/SuSuInterActs before continuing.'
    )

## 2. Direct multipart comparison

This is the primary baseline decision. All three models receive their saved audio-codebook configuration but share the same causal multipart targets, fixed schedule, validation clips, greedy decoding, and previous-GT-anchor persistence reference.

In [ ]:
if RUN_MULTIPART:
    command = [
        sys.executable,
        str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_step1_multipart_comparison.py'),
        '--output_dir', str(OUTPUT_DIR),
        '--teacher_max_clips', str(MULTIPART_TEACHER_MAX_CLIPS),
        '--rollout_max_clips', str(MULTIPART_ROLLOUT_MAX_CLIPS),
        '--rollout_batch_size', str(MULTIPART_ROLLOUT_BATCH),
        '--subset_seed', str(SUBSET_SEED),
    ]
    print(' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('multipart evaluation skipped; reading existing outputs')

## 3. Released SentiAvatar Step 1 in its native representation

The released model has no motion seed and explicitly declares interval-end frames `[frame_3]`, `[frame_7]`, ... for step four. It is scored at those native times rather than shifted onto our seed-at-0 / anchors-at-4 schedule. Coverage and length errors are reported so malformed or truncated plans cannot receive an artificially flattering matched-only score.

In [ ]:
if RUN_RELEASED_SENTIAVATAR:
    command = [
        sys.executable,
        str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_sentiavatar_legacy_step1.py'),
        '--checkpoint', str(CHECKPOINTS['released_sentiavatar']),
        '--data_dir', str(DATA_DIR),
        '--output_dir', str(OUTPUT_DIR),
        '--max_clips', str(LEGACY_ROLLOUT_MAX_CLIPS),
        '--batch_size', str(LEGACY_BATCH),
        '--subset_seed', str(SUBSET_SEED),
    ]
    print(' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('released SentiAvatar evaluation skipped; reading existing outputs')

## 4. Results

Use the multipart table to choose our checkpoint. Use the cross-representation table only for within-system margins over persistence, coverage, horizon collapse, and efficiency—not as a direct ID-space leaderboard.

In [ ]:
multipart_contracts = pd.read_csv(OUTPUT_DIR / 'multipart_contracts.csv')
multipart_teacher = pd.read_csv(OUTPUT_DIR / 'multipart_teacher_forced.csv')
multipart_rollout = pd.read_csv(OUTPUT_DIR / 'multipart_generated_rollout.csv')
multipart_slots = pd.read_csv(OUTPUT_DIR / 'multipart_generated_rollout_per_slot.csv')
multipart_horizon = pd.read_csv(OUTPUT_DIR / 'multipart_generated_rollout_horizon.csv')
display(multipart_contracts)
display(multipart_teacher[[
    'checkpoint', 'accuracy', 'top5_accuracy', 'cross_entropy', 'perplexity',
    'q0_accuracy', 'q1_accuracy', 'q2_accuracy', 'q3_accuracy',
]])
display(multipart_rollout[[
    'checkpoint', 'clips', 'accuracy', 'q0_accuracy', 'q1_accuracy',
    'q2_accuracy', 'q3_accuracy', 'teacher_forced_accuracy_same_subset',
    'previous_copy_accuracy_same_subset', 'accuracy_drop_from_teacher_forcing',
    'accuracy_margin_over_previous_copy', 'expected_calibration_error_10bin',
    'mean_confidence', 'elapsed_seconds',
]])

In [ ]:
comparison_rows = []
for row in multipart_rollout.to_dict(orient='records'):
    uniform = 1 / 512
    comparison_rows.append({
        'system': row['checkpoint'],
        'representation': 'causal multipart 16x512',
        'clips': int(row['clips']),
        'strict_accuracy': row['accuracy'],
        'q0_accuracy': row['q0_accuracy'],
        'previous_copy_accuracy': row['previous_copy_accuracy_same_subset'],
        'margin_over_copy': row['accuracy_margin_over_previous_copy'],
        'persistence_ratio': row['accuracy'] / row['previous_copy_accuracy_same_subset'],
        'normalized_accuracy_above_uniform': (row['accuracy'] - uniform) / (1 - uniform),
        'coverage': 1.0,
        'seed_protocol': 'known GT/previous seed',
    })
if RUN_RELEASED_SENTIAVATAR or (OUTPUT_DIR / 'legacy_sentiavatar_rollout.csv').exists():
    legacy = pd.read_csv(OUTPUT_DIR / 'legacy_sentiavatar_rollout.csv').iloc[0]
    comparison_rows.append({
        'system': 'released_sentiavatar_step1',
        'representation': 'legacy whole-body 4x512',
        'clips': int(legacy['clips']),
        'strict_accuracy': legacy['accuracy'],
        'q0_accuracy': legacy['q0_accuracy'],
        'previous_copy_accuracy': legacy['previous_copy_accuracy'],
        'margin_over_copy': legacy['accuracy_margin_over_previous_copy'],
        'persistence_ratio': legacy['persistence_ratio'],
        'normalized_accuracy_above_uniform': legacy['normalized_accuracy_above_uniform'],
        'coverage': legacy['coverage'],
        'seed_protocol': legacy['seed_protocol'],
    })
cross_representation = pd.DataFrame(comparison_rows)
display(cross_representation)
cross_representation.to_csv(OUTPUT_DIR / 'cross_representation_summary.csv', index=False)

In [ ]:
multipart_horizon['horizon_bin'] = pd.cut(
    multipart_horizon['relative_horizon'],
    bins=[-1e-9, .25, .5, .75, 1.0],
    labels=['0-25%', '25-50%', '50-75%', '75-100%'],
    include_lowest=True,
)
horizon_summary = multipart_horizon.groupby(
    ['checkpoint', 'horizon_bin'], observed=True, as_index=False
)[['accuracy', 'q0_accuracy', 'confidence', 'entropy']].mean()
display(horizon_summary)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for checkpoint, group in horizon_summary.groupby('checkpoint'):
    axes[0].plot(group['horizon_bin'].astype(str), group['accuracy'] * 100, marker='o', label=checkpoint)
    axes[1].plot(group['horizon_bin'].astype(str), group['q0_accuracy'] * 100, marker='o', label=checkpoint)
axes[0].set_ylabel('All-slot strict accuracy (%)')
axes[1].set_ylabel('q0 strict accuracy (%)')
for axis in axes:
    axis.set_xlabel('Relative utterance horizon')
    axis.grid(alpha=.25)
    axis.legend()
fig.tight_layout()
plt.show()

## 5. Decision rules

Choose the multipart baseline using generated rollout, not teacher-forced CE alone. A checkpoint passes the standalone planner gate only if it improves over the earlier q0-only rollout, reduces horizon collapse and calibration error, and preferably beats previous-anchor copying on the identical subset.

The released SentiAvatar row is contextual: compare its persistence ratio, malformed-output coverage, horizon trend, and eventual decoded motion. Do not claim one system wins because its raw token percentage is larger across different codebooks.

After selecting the multipart checkpoint, the next fair experiment is a common raw-motion/end-to-end table with each planner's own codec and frozen Step 2, reporting codec oracle error, incremental anchor damage, infilled motion quality, latency, and anchor budget.